# **Inference Cost Benchmark**

In [1]:
%pip install -q torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 194.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 237.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 245.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 61.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 84.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 136.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 196.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 126.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 121.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 122.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 122.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 146.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Section 1: Config**

In [3]:
import os
import io
import json
import time
import zipfile
import contextlib
import numpy as np
import pandas as pd
import torch
import torchvision
from torchvision.transforms import v2, InterpolationMode
from IPython.display import display

drive_dir = '/content/drive/MyDrive/projectMushroom'
os.chdir(drive_dir)
import src.ensembleVoting as ensembleVoting
import src.reproducibility as reproducibility
import models.models as models

SEED = 42
BENCH_DATASETS = ['MushroomA', 'MushroomB']
N_FOLDS = 5
WEIGHT_TYPE = 'best'
BATCH_SIZES = [1, 32]
INFER_BATCH_SIZE = 32
BENCH_CSV = f'{drive_dir}/addition/latex_tables_benchmark.csv'
BENCH_ENV_JSON = f'{drive_dir}/addition/latex_tables_benchmark_env.json'

MODELS = [
    ('ResNet-50',       'modelWeights',   'resnet50_224_r1_v1',                'ResNet50',           None),
    ('ResNeXt-50',      'modelWeights',   'resnext50_224_r1_v1',               'ResNeXt50',          None),
    ('ConvNeXt-T',      'modelWeights',   'convnext_tiny_224_r1_v1',           'ConvNeXt-Tiny',      None),
    ('InceptionNeXt-T', 'modelWeights',   'inceptionnext_tiny_224_r1_v1',      'InceptionNeXt-Tiny', None),
    ('DeiT-S',          'modelWeights',   'deit_small_224_r1_v1',              'DeiT-Small',         None),
    ('MarisaNeXt-T',    'myModelWeights', 'convnext_tiny_modify_D6_224_r1_v1', 'ConvNeXt-Tiny',      'D6'),
]

def model_dir(weights_root, name, ds):
    return f'{drive_dir}/{weights_root}/{ds}/{name}_{ds}'

MODEL_SPECS = {label: spec for label, *spec in MODELS}

## **Section 2: Load Logs**

In [4]:
def read_csv(path, **kwargs):
    if not os.path.exists(path):
        raise FileNotFoundError(f'[Error] File not found: {path}')
    return pd.read_csv(path, **kwargs)

def read_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'[Error] File not found: {path}')
    with open(path) as f:
        return json.load(f)

def quiet(fn, *args, **kwargs):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

def check_equal(name, expected, actual):
    if expected != actual:
        raise AssertionError(f'[Error] {name}: saved = {actual}, recomputed = {expected}')

LOGS = {}
for label, weights_root, name, *_ in MODELS:
    for ds in BENCH_DATASETS:
        d = model_dir(weights_root, name, ds)
        y_true, _, idx_to_class = quiet(ensembleVoting.load_test_records, d, weight_type=WEIGHT_TYPE)
        LOGS[(label, ds)] = {
            'y_true': y_true,
            'idx_to_class': idx_to_class,
            'fold_probs': quiet(ensembleVoting.load_fold_probs, d, N_FOLDS, weight_type=WEIGHT_TYPE),
            'best_fold': read_json(f'{d}/summary/best_fold_info.json')['best_fold_num'],
            'gflops': float(read_csv(f'{d}/summary/model_summary.csv').iloc[0]['GFLOPs']),
        }
        print(f'[Logs] {label} {ds}: loaded test logs')

[Logs] ResNet-50 MushroomA: loaded test logs
[Logs] ResNet-50 MushroomB: loaded test logs
[Logs] ResNeXt-50 MushroomA: loaded test logs
[Logs] ResNeXt-50 MushroomB: loaded test logs
[Logs] ConvNeXt-T MushroomA: loaded test logs
[Logs] ConvNeXt-T MushroomB: loaded test logs
[Logs] InceptionNeXt-T MushroomA: loaded test logs
[Logs] InceptionNeXt-T MushroomB: loaded test logs
[Logs] DeiT-S MushroomA: loaded test logs
[Logs] DeiT-S MushroomB: loaded test logs
[Logs] MarisaNeXt-T MushroomA: loaded test logs
[Logs] MarisaNeXt-T MushroomB: loaded test logs


## **Section 3: Benchmark**

In [5]:
device = torch.device('cuda')
assert torch.cuda.is_available(), '[Error] GPU runtime is required.'
reproducibility.training_seed(seed=SEED)
GPU = torch.cuda.get_device_name(0)
ENV = {
    'gpu': GPU,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'cudnn_deterministic': torch.backends.cudnn.deterministic,
    'cudnn_benchmark': torch.backends.cudnn.benchmark,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'float32_matmul_precision': torch.get_float32_matmul_precision(),
    'started': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
for key, value in ENV.items():
    print(f'{key}: {value}')

def build_model(model_name, modify_type, num_classes):
    if modify_type is None:
        return models.build_baseline_model(model_name=model_name, num_classes=num_classes, device='cpu', freeze_backbone=False)
    return models.build_modify_convnext(model_name=model_name, num_classes=num_classes, device='cpu',
                                        target_stages=[0, 1, 2, 3], freeze_backbone=False, modify_type=modify_type)

def checkpoint_path(label, ds, k):
    weights_root, name, _, _ = MODEL_SPECS[label]
    return f'{model_dir(weights_root, name, ds)}/fold_{k}/{name}_{ds}_best_weight.pth'

def load_checkpoint(path, attempts=3):
    size = os.path.getsize(path)
    for attempt in range(1, attempts + 1):
        with open(path, 'rb') as f:
            data = f.read()
        try:
            if len(data) != size:
                raise EOFError(f'read {len(data)} of {size} bytes')
            return torch.load(io.BytesIO(data), map_location='cpu')
        except (EOFError, RuntimeError) as e:
            print(f'[Warning] {path}: attempt {attempt}/{attempts} failed ({type(e).__name__}: {e})')
            time.sleep(10 * attempt)
    raise RuntimeError(f'[Error] Cannot load {path} ({size} bytes) after {attempts} attempts')

def load_fold_models(label, ds, num_classes):
    weights_root, name, model_name, modify_type = MODEL_SPECS[label]
    fold_models = {}
    for k in range(1, N_FOLDS + 1):
        model = build_model(model_name, modify_type, num_classes)
        ckpt = load_checkpoint(checkpoint_path(label, ds, k))
        model.load_state_dict(ckpt['model_state_dict'], strict=True)
        fold_models[k] = model.eval()
        del ckpt
    return fold_models

def test_tensor(model, ds):
    split_dir = f'/content/bench/{ds}'
    if not os.path.exists(split_dir):
        with zipfile.ZipFile(f'{drive_dir}/data/{ds}/{ds}_SPLIT.zip', 'r') as z:
            z.extractall(split_dir)
    cfg = models.get_transform_config(model)
    interp = getattr(InterpolationMode, cfg['interpolation'].upper())
    transform = v2.Compose([
        v2.Resize(int(cfg['img_size'][0] / cfg['crop_pct']), interpolation=interp, max_size=None, antialias=True),
        v2.CenterCrop(cfg['img_size']),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=cfg['mean'], std=cfg['std']),
    ])
    dataset = torchvision.datasets.ImageFolder(f'{split_dir}/test', transform=transform)
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)
    return torch.cat([x for x, _ in loader]), len(dataset.classes)

def run_pass(fold_models, X, batch_size, record):
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    batch_ms, batch_n, probs = [], [], []
    with torch.inference_mode():
        for i in range(0, len(X), batch_size):
            x = X[i:i + batch_size].to(device, non_blocking=True)
            torch.cuda.synchronize()
            start.record()
            p = torch.stack([torch.softmax(m(x), dim=1) for m in fold_models]).mean(dim=0)
            end.record()
            torch.cuda.synchronize()
            if record:
                batch_ms.append(start.elapsed_time(end))
                batch_n.append(len(x))
                probs.append(p.float().cpu())
    return np.array(batch_ms), np.array(batch_n), (torch.cat(probs).numpy() if record else None)

def state_dict_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 2**20

Training Seed set to: 42
gpu: NVIDIA L4
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
cuda: 12.8
cudnn: 91002
cudnn_deterministic: True
cudnn_benchmark: False
deterministic_algorithms: True
float32_matmul_precision: highest
started: 2026-10-14 03:12:13 UTC


In [6]:
bad_checkpoints = [(path, os.path.getsize(path)) for ds in BENCH_DATASETS for label, *_ in MODELS for k in range(1, N_FOLDS + 1)
                   for path in [checkpoint_path(label, ds, k)] if not zipfile.is_zipfile(path)]
if bad_checkpoints:
    raise AssertionError('[Error] Checkpoints are not complete torch files:\n' + '\n'.join(f'{p} ({b} bytes)' for p, b in bad_checkpoints))
print(f'[Logs] All {len(BENCH_DATASETS) * len(MODELS) * N_FOLDS} checkpoints are complete torch files')

BENCH, BENCH_MISMATCH = {}, {}
for ds in BENCH_DATASETS:
    for label, _, _, model_name, modify_type in MODELS:
        run = LOGS[(label, ds)]
        k = run['best_fold']
        saved = {'Single': run['fold_probs'][k - 1].argmax(axis=1), 'Ensemble': ensembleVoting.soft_voting(run['fold_probs']).argmax(axis=1)}

        probe = build_model(model_name, modify_type, num_classes=len(run['idx_to_class']))
        X, num_classes = test_tensor(probe, ds)
        del probe
        check_equal(f'{ds} {label} number of classes', len(run['idx_to_class']), num_classes)
        check_equal(f'{ds} {label} benchmark images', len(run['y_true']), len(X))
        cpu_models = load_fold_models(label, ds, num_classes)

        for mode, folds in [('Single', [k]), ('Ensemble', list(range(1, N_FOLDS + 1)))]:
            for bs in BATCH_SIZES:
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                fold_models = [cpu_models[f].to(device) for f in folds]
                run_pass(fold_models, X, bs, record=False)
                batch_ms, batch_n, probs = run_pass(fold_models, X, bs, record=True)
                peak_mb = torch.cuda.max_memory_allocated() / 2**20

                mismatch = int((probs.argmax(axis=1) != saved[mode]).sum())
                if mismatch and bs == INFER_BATCH_SIZE:
                    raise AssertionError(f'[Error] {ds} {label} {mode} bs={bs}: {mismatch} predictions differ from the saved probabilities')
                BENCH_MISMATCH[(ds, label, mode, bs)] = mismatch

                per_img = batch_ms / batch_n
                BENCH[(ds, label, mode, bs)] = {
                    'n_models': len(folds), 'n_iterations': len(batch_ms), 'n_images': int(batch_n.sum()),
                    'params_m': sum(p.numel() for m in fold_models for p in m.parameters()) / 1e6,
                    'storage_mb': sum(state_dict_mb(m) for m in fold_models),
                    'gflops': run['gflops'] * len(folds),
                    'latency_median_ms': np.median(per_img), 'latency_mean_ms': per_img.mean(),
                    'latency_sd_ms': per_img.std(ddof=1), 'latency_p95_ms': np.percentile(per_img, 95),
                    'throughput_img_s': batch_n.sum() / (batch_ms.sum() / 1000), 'peak_memory_mb': peak_mb,
                }
                print(f'[Logs] {ds} | {label} | {mode} | bs={bs} | median {np.median(per_img):.3f} ms/img | peak {peak_mb:.0f} MB | mismatch {mismatch}')
                for m in fold_models:
                    m.to('cpu')
                del fold_models
                torch.cuda.empty_cache()
        del X, cpu_models

results = pd.DataFrame([{'dataset': ds, 'model': l, 'mode': m, 'batch_size': bs, **v, 'mismatch': BENCH_MISMATCH[(ds, l, m, bs)], 'gpu': GPU}
                        for (ds, l, m, bs), v in BENCH.items()])
if os.path.exists(BENCH_CSV):
    previous = BENCH_CSV.replace('.csv', time.strftime('_previous_%Y%m%d_%H%M%S.csv', time.localtime(os.path.getmtime(BENCH_CSV))))
    os.rename(BENCH_CSV, previous)
    print(f'[Logs] Moved the previous benchmark to {previous}')
results.to_csv(BENCH_CSV, index=False)
ENV['finished'] = time.strftime('%Y-%m-%d %H:%M:%S %Z')
with open(BENCH_ENV_JSON, 'w') as f:
    json.dump(ENV, f, indent=2)
print(f'[Logs] Saved {len(results)} rows to {BENCH_CSV}')
print(f'[Logs] Saved the environment to {BENCH_ENV_JSON}')

[Logs] All 60 checkpoints are complete torch files


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

[Logs] MushroomA | ResNet-50 | Single | bs=1 | median 7.158 ms/img | peak 133 MB | mismatch 0
[Logs] MushroomA | ResNet-50 | Single | bs=32 | median 1.435 ms/img | peak 460 MB | mismatch 0
[Logs] MushroomA | ResNet-50 | Ensemble | bs=1 | median 36.569 ms/img | peak 504 MB | mismatch 0
[Logs] MushroomA | ResNet-50 | Ensemble | bs=32 | median 7.354 ms/img | peak 830 MB | mismatch 0


model.safetensors: reconstructing file:   0%|          |  0.00B /  100MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

[Logs] MushroomA | InceptionNeXt-T | Single | bs=1 | median 9.408 ms/img | peak 147 MB | mismatch 0
[Logs] MushroomA | InceptionNeXt-T | Single | bs=32 | median 1.423 ms/img | peak 557 MB | mismatch 0
[Logs] MushroomA | InceptionNeXt-T | Ensemble | bs=1 | median 47.437 ms/img | peak 545 MB | mismatch 0
[Logs] MushroomA | InceptionNeXt-T | Ensemble | bs=32 | median 7.346 ms/img | peak 954 MB | mismatch 0


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

[Logs] MushroomA | DeiT-S | Single | bs=1 | median 11.594 ms/img | peak 121 MB | mismatch 0
[Logs] MushroomA | DeiT-S | Single | bs=32 | median 1.457 ms/img | peak 245 MB | mismatch 0
[Logs] MushroomA | DeiT-S | Ensemble | bs=1 | median 57.870 ms/img | peak 452 MB | mismatch 0
[Logs] MushroomA | DeiT-S | Ensemble | bs=32 | median 7.740 ms/img | peak 577 MB | mismatch 0
[Logs] MushroomA | MarisaNeXt-T | Single | bs=1 | median 13.348 ms/img | peak 169 MB | mismatch 0
[Logs] MushroomA | MarisaNeXt-T | Single | bs=32 | median 2.460 ms/img | peak 1009 MB | mismatch 0
[Logs] MushroomA | MarisaNeXt-T | Ensemble | bs=1 | median 67.566 ms/img | peak 598 MB | mismatch 0
[Logs] MushroomA | MarisaNeXt-T | Ensemble | bs=32 | median 12.748 ms/img | peak 1437 MB | mismatch 0
[Logs] MushroomB | ResNet-50 | Single | bs=1 | median 7.396 ms/img | peak 134 MB | mismatch 0
[Logs] MushroomB | ResNet-50 | Single | bs=32 | median 1.452 ms/img | peak 460 MB | mismatch 0
[Logs] MushroomB | ResNet-50 | Ensemble 

## **Section 4: Saved Results**

In [7]:
saved = read_csv(BENCH_CSV, float_precision='round_trip')
check_equal('saved columns', list(results.columns), list(saved.columns))
check_equal('saved rows', len(results), len(saved))
for col in results.columns:
    if not pd.api.types.is_numeric_dtype(results[col]):
        check_equal(f'saved {col}', results[col].tolist(), saved[col].tolist())
    elif not np.array_equal(results[col].to_numpy(dtype=float), saved[col].to_numpy(dtype=float)):
        raise AssertionError(f'[Error] saved {col} differs from the measured values')
modified = time.strftime('%Y-%m-%d %H:%M:%S %Z', time.localtime(os.path.getmtime(BENCH_CSV)))
print(f'[Logs] {BENCH_CSV}: {os.path.getsize(BENCH_CSV):,} bytes, modified {modified}')
print(f'[Logs] All {len(saved)} saved rows are identical to the measured values')
print(json.dumps(read_json(BENCH_ENV_JSON), indent=2))

summary = saved.pivot_table(index=['dataset', 'model', 'mode'], columns='batch_size', sort=False,
                            values=['latency_median_ms', 'latency_p95_ms', 'throughput_img_s', 'peak_memory_mb', 'mismatch'])
display(summary.round(2))

[Logs] /content/drive/MyDrive/projectMushroom/addition/latex_tables_benchmark.csv: 9,768 bytes, modified 2026-10-14 03:58:19 UTC
[Logs] All 48 saved rows are identical to the measured values
{
  "gpu": "NVIDIA L4",
  "torch": "2.10.0+cu128",
  "torchvision": "0.25.0+cu128",
  "cuda": "12.8",
  "cudnn": 91002,
  "cudnn_deterministic": true,
  "cudnn_benchmark": false,
  "deterministic_algorithms": true,
  "float32_matmul_precision": "highest",
  "started": "2026-10-14 03:12:13 UTC",
  "finished": "2026-10-14 03:58:19 UTC"
}


latency_median_ms        latency_p95_ms  \
batch_size                                        1      32             1    
dataset   model           mode                                               
MushroomA ResNet-50       Single                7.16   1.44           7.51   
                          Ensemble             36.57   7.35          37.94   
          ResNeXt-50      Single                7.21   1.92           7.45   
                          Ensemble             36.73   9.82          37.74   
          ConvNeXt-T      Single               10.79   1.84          11.07   
                          Ensemble             54.83   9.99          56.13   
          InceptionNeXt-T Single                9.41   1.42           9.71   
                          Ensemble             47.44   7.35          48.69   
          DeiT-S          Single               11.59   1.46          12.10   
                          Ensemble             57.87   7.74          59.18   
          MarisaNeXt-T    Single               13.35   2.46          13.72   
                          Ensemble             67.57  12.75          68.56   
MushroomB ResNet-50       Single                7.40   1.45           7.77   
                          Ensemble             37.54   7.40          38.75   
          ResNeXt-50      Single                7.43   1.92           7.64   
                          Ensemble             37.10   9.84          39.00   
          ConvNeXt-T      Single               11.00   1.80          11.30   
                          Ensemble             54.65   9.84          55.72   
          InceptionNeXt-T Single                9.48   1.42           9.83   
                          Ensemble             47.36   7.28          48.88   
          DeiT-S          Single               11.56   1.40          12.16   
                          Ensemble             58.13   7.55          59.35   
          MarisaNeXt-T    Single               13.52   2.43          13.85   
                          Ensemble             67.60  12.93          68.94   

                                          throughput_img_s          \
batch_size                             32               1       32   
dataset   model           mode                                       
MushroomA ResNet-50       Single     1.45           138.97  698.05   
                          Ensemble   7.37            27.25  136.23   
          ResNeXt-50      Single     1.94           138.31  521.06   
                          Ensemble   9.85            27.15  101.99   
          ConvNeXt-T      Single     1.97            92.41  542.22   
                          Ensemble  10.22            18.20   99.85   
          InceptionNeXt-T Single     1.45           105.96  700.52   
                          Ensemble   7.37            21.02  136.26   
          DeiT-S          Single     1.51            85.85  691.67   
                          Ensemble   7.91            17.25  129.05   
          MarisaNeXt-T    Single     2.52            74.73  407.16   
                          Ensemble  12.85            14.79   78.55   
MushroomB ResNet-50       Single     1.47           134.40  689.00   
                          Ensemble   7.43            26.59  135.41   
          ResNeXt-50      Single     1.95           134.25  519.98   
                          Ensemble   9.86            26.85  101.87   
          ConvNeXt-T      Single     1.94            90.76  551.45   
                          Ensemble   9.95            18.27  101.80   
          InceptionNeXt-T Single     1.46           104.88  705.12   
                          Ensemble   7.32            21.04  137.56   
          DeiT-S          Single     1.48            85.96  712.99   
                          Ensemble   7.65            17.16  132.66   
          MarisaNeXt-T    Single     2.48            73.79  411.22   
                          Ensemble  13.01            14.77   77.63   

                                   pe